# 2.2 Measures of Central Tendency

**Goal:** select a representative center that matches the data's scale and the question being asked.

**Prerequisites:** frequency distributions, arithmetic, logarithms, and measurement scales. **Runtime:** Python 3.10+, standard library only. Synthetic examples with saved outputs; run cells in order.

## Contents
1. Arithmetic mean
2. Weighted mean
3. Geometric mean
4. Harmonic mean
5. Median
6. Mode
7. Relationship between mean, median, and mode
8. Imputation application, practice, and solutions

## 1. Arithmetic Mean

For $n>0$ numerical observations,
$$\bar x=\frac1n\sum_{i=1}^n x_i.$$
For a complete finite population, the corresponding parameter is often denoted $\mu$. The arithmetic mean uses every value and has the same units as the data. It is appropriate when additive totals and equal numerical differences are meaningful.

**Worked example:** values 2, 4, 6 have mean 4. Replacing 6 by 60 changes the mean to 22. Extreme values can strongly affect the mean; this sensitivity can be desirable when the total burden matters, such as average cost, but unhelpful when describing a typical skewed case.

The mean balances signed deviations: $\sum_i(x_i-\bar x)=0$. It also minimizes $\sum_i(x_i-c)^2$ over constants $c$. This connects the mean to squared-error prediction. Means of arbitrary nominal codes lack substantive meaning.

In [1]:
import math
import statistics
from collections import Counter
xs = [2, 4, 6]
mean = statistics.mean(xs)
print("Mean:", mean, "sum of deviations:", sum(x-mean for x in xs))
print("With a large observation:", statistics.mean([2, 4, 60]))
for center in [3, 4, 5]:
    print("Center:", center, "sum squared error:", sum((x-center)**2 for x in xs))
assert sum(x-mean for x in xs) == 0

Mean: 4 sum of deviations: 0
With a large observation: 22
Center: 3 sum squared error: 11
Center: 4 sum squared error: 8
Center: 5 sum squared error: 11


## 2. Weighted Mean

With nonnegative weights $w_i$ whose sum is positive,
$$\bar x_w=\frac{\sum_iw_ix_i}{\sum_iw_i}.$$
Weights can represent frequencies, exposure amounts, credits, or a justified weighting design; those meanings are not interchangeable. Multiplying all weights by the same positive constant leaves the result unchanged.

**Worked example:** scores 80 and 90 with course credits 1 and 3 have weighted mean $(80+270)/4=87.5$. An unweighted mean of 85 gives each course equal influence instead of each credit.

To pool group means, weight by group sizes: 9/10 correct and 50/100 correct give total accuracy $59/110\approx0.5364$, not the equal-group average of 0.9 and 0.5. An equal-group average can be intentional, but answers a different question.

In [2]:
def weighted_mean(values, weights):
    if len(values) != len(weights) or not values:
        raise ValueError("Provide equally sized nonempty sequences")
    if any(w < 0 or not math.isfinite(w) for w in weights) or sum(weights) <= 0:
        raise ValueError("Weights must be finite, nonnegative, with positive total")
    return sum(x*w for x, w in zip(values, weights))/sum(weights)

print("Credit-weighted score:", weighted_mean([80, 90], [1, 3]))
print("Pooled accuracy:", weighted_mean([0.9, 0.5], [10, 100]))
print("Equal-group mean accuracy:", statistics.mean([0.9, 0.5]))
assert weighted_mean([80, 90], [1, 3]) == 87.5

Credit-weighted score: 87.5
Pooled accuracy: 0.5363636363636364
Equal-group mean accuracy: 0.7


## 3. Geometric Mean

For strictly positive values,
$$G=\left(\prod_i x_i\right)^{1/n}=\exp\left(\frac1n\sum_i\ln x_i\right).$$
It summarizes multiplicative factors and proportional growth. The log formulation avoids forming a potentially huge or tiny product. Standard log-based use requires positivity; do not silently take logs of zeros or negative values.

**Worked example:** an amount grows 20%, then falls 20%. The growth factors are 1.2 and 0.8, with product 0.96. The equivalent per-period factor is $\sqrt{0.96}\approx0.979796$, so the geometric average growth rate is approximately $-2.0204\%$, not the arithmetic average of the signed rates (0%).

For growth rates $r_i$, average the factors $1+r_i$ geometrically, then subtract 1. This calculation assumes equal time intervals and no interfering cash flows for an investment-style interpretation. A 100% loss creates a zero factor and needs separate handling.

In [3]:
factors = [1.2, 0.8]
geometric = math.exp(statistics.mean(math.log(x) for x in factors))
print("Final multiplier:", math.prod(factors))
print("Equivalent per-period factor:", geometric)
print("Equivalent per-period growth (%):", 100*(geometric-1))
assert math.isclose(geometric**2, 0.96)

Final multiplier: 0.96
Equivalent per-period factor: 0.9797958971132712
Equivalent per-period growth (%): -2.0204102886728803


## 4. Harmonic Mean

For strictly positive values,
$$H=\frac{n}{\sum_i1/x_i}.$$
The harmonic mean is appropriate for averaging rates when each rate applies to the same amount of the numerator quantity, such as equal-distance legs of a journey. It is sensitive to small positive values. A zero rate over a positive-distance segment would imply infinite travel time, so the usual finite-rate formula cannot simply ignore it.

**Worked example:** travel 60 km at 30 km/h and another 60 km at 60 km/h. Total time is 2+1=3 hours; average speed is $120/3=40$ km/h, the harmonic mean of 30 and 60. If you instead travel for equal **time** at each speed, average speed is the arithmetic mean, 45 km/h.

For positive values, $H\le G\le\bar x$, with equality when all values are equal. This inequality does not tell you which mean answers your question; the denominator/exposure structure does.

In [4]:
speeds = [30, 60]
distances = [60, 60]
total_distance = sum(distances)
total_time = sum(d/v for d, v in zip(distances, speeds))
harmonic = len(speeds)/sum(1/v for v in speeds)
print("Distance/time average speed:", total_distance/total_time)
print("Harmonic mean:", harmonic, "arithmetic mean:", statistics.mean(speeds))
assert math.isclose(harmonic, 40)
assert harmonic <= statistics.geometric_mean(speeds) <= statistics.mean(speeds)

Distance/time average speed: 40.0
Harmonic mean: 40.0 arithmetic mean: 45


## 5. Median

Sort observations first. With odd $n$, the median is the middle observation. With even $n$ and numerical values, it is commonly the mean of the two middle observations. It is less affected by extreme magnitudes than the mean.

**Worked example:** 1, 2, 3, 4, 100 have median 3 and mean 22. Replacing 100 with 1000 leaves the median unchanged. For ordinal categories, report a median category or central categories instead of assuming numeric gaps are meaningful.

A median minimizes $\sum_i|x_i-c|$ over constants $c$. For even $n$, every value between the two middle observations minimizes this absolute-error objective; their midpoint is a conventional single choice.

## 6. Mode

A mode is a most frequent value or category. There can be one mode, several tied modes, or no uniquely informative mode. For continuous measurements, exact repeats depend on rounding; a histogram's **modal class** is a bin with the highest frequency and depends on binning. A density mode is a peak of an estimated or theoretical density, not necessarily an observed value.

**Worked example:** red, blue, red, green, blue have two modes, red and blue. Reporting only the first mode encountered hides the tie.

In [5]:
xs = [1, 2, 3, 4, 100]
print("Mean:", statistics.mean(xs), "Median:", statistics.median(xs))
for center in [2, 3, 4]:
    print("Center:", center, "total absolute error:", sum(abs(x-center) for x in xs))
colors = ["red", "blue", "red", "green", "blue"]
print("All tied modes:", sorted(statistics.multimode(colors)))
assert statistics.median(xs) == 3

Mean: 22 Median: 3
Center: 2 total absolute error: 102
Center: 3 total absolute error: 101
Center: 4 total absolute error: 102
All tied modes: ['blue', 'red']


## 7. Relationship between Mean, Median and Mode

For a distribution symmetric about a center with an existing finite mean, the mean equals that center and the center is a median. If it is also unimodal with a unique peak at the center, the mode agrees. Symmetry alone does not require a single central mode: a symmetric two-peaked distribution is a counterexample.

A common heuristic for many unimodal right-skewed distributions is mode < median < mean; left-skewed examples often reverse the order. These are not universal theorems for all skewed or multimodal distributions. Do not infer a distribution's complete shape from three summaries alone.

The empirical Pearson relation
$$\text{Mode}\approx3\,\text{Median}-2\,\text{Mean}$$
is a rough historical approximation for some moderately skewed unimodal distributions, not an identity, a robust mode estimator, or a rule for arbitrary datasets.

| Question | Possible summary | Main caution |
|---|---|---|
| Average additive amount | Arithmetic mean | Sensitive to extremes |
| Combine group summaries | Weighted mean | Weights must match the estimand |
| Equivalent repeated growth factor | Geometric mean | Positive factors; specify intervals |
| Rate over equal distances | Harmonic mean | Check numerator/exposure assumptions |
| Typical skewed observation | Median | Does not preserve the additive total |
| Most common category | Mode(s) | Ties and coding/rounding matter |

## 8. AI/ML application: impute from training data only

Mean imputation is simple but sensitive to extreme values. Median imputation is more resistant to their magnitude; mode imputation can fill a categorical field. All can suppress variability or distort relationships, so missingness must be investigated and the strategy evaluated. Learn imputation values on training observations only and apply them unchanged to validation/test data.

In [6]:
train = [10.0, 12.0, None, 14.0, 100.0]
held_out = [None, 1000.0]
observed_train = [x for x in train if x is not None]
mean_fill = statistics.mean(observed_train)
median_fill = statistics.median(observed_train)
print("Training-only mean:", mean_fill, "median:", median_fill)
print("Held-out mean-imputed:", [mean_fill if x is None else x for x in held_out])
print("Held-out median-imputed:", [median_fill if x is None else x for x in held_out])
assert median_fill == 13

Training-only mean: 34.0 median: 13.0
Held-out mean-imputed: [34.0, 1000.0]
Held-out median-imputed: [13.0, 1000.0]


## Practice

1. Find the mean, median, and all modes of $2,2,4,8$.
2. Scores 70, 80, 90 have weights 1, 2, 1. Find their weighted mean.
3. An amount grows 10% then 21%. Find the equivalent per-period growth rate.
4. Travel equal distances at 40 and 60 km/h. Find average speed. What if travel times are equal?
5. Why is a geometric mean of signed percentage rates usually the wrong calculation?
6. Does a symmetric distribution always have a unique mode at its center?
7. A validation value is 10000. Should it affect a training-set median used to impute validation missing values?

## Solutions

1. Mean $16/4=4$; median $(2+4)/2=3$; unique mode 2.
2. $(70+160+90)/4=80$.
3. Factor $\sqrt{1.10\times1.21}=\sqrt{1.331}\approx1.15369$; rate about 15.369% per period.
4. Harmonic mean $2/(1/40+1/60)=48$ km/h. Equal times give arithmetic mean 50 km/h.
5. Multiplicative accumulation applies to $1+r_i$, not $r_i$, and negative/zero rates are outside the usual positive-value geometric mean domain. Compute the mean of positive factors and subtract 1.
6. No. Symmetric bimodal distributions can have two off-center peaks. Agreement of all three centers needs stronger conditions.
7. No. The imputation statistic must be fit on training data alone; validation data evaluate the fitted procedure.

**Readiness check:** choose a mean from the aggregation structure, report mode ties, and explain the difference between robustness and the quantity you want to estimate.

**Next:** [2.3 Measures of Dispersion](3_Measures-of-Dispersion.ipynb).